# BGC Donor Data — Source Intake & Standardization

## Objective
Standardize each raw Excel source file and export cleaned versions before combining.

## Week 1 Focus
- Confirm folder structure
- Detect raw files
- Build source registry
- Process one file fully
- Validate export structure

## Data Handling Rules
- Raw files are read-only
- All transformations happen in working copy
- Each source exported separately
- No combining yet

## Step 1 — Imports & Project Paths

**Purpose:** Set up libraries and define reliable folder paths for raw and cleaned data.  
**What I’m looking for:** `RAW_DIR` and `CLEAN_DIR` both exist (or I create `CLEAN_DIR`).  
**How I’ll use it next:** All file loading/saving will reference these paths to avoid broken paths.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project root (because notebook is in /notebooks)
BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" /"cleaned"

# Confirm folders exist
print("BASE_DIR:", BASE_DIR)
print("RAW_DIR exists:", RAW_DIR.exists(), "|", RAW_DIR)
print("CLEAN_DIR exists:", CLEAN_DIR.exists(), "|", CLEAN_DIR)

# Create cleaned folder if missing
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("CLEAN_DIR now exists:", CLEAN_DIR.exists())

BASE_DIR: /home/ce5b4deb-dc9a-4dd7-9a99-cd66f2b354ff/BGC_donor_project
RAW_DIR exists: True | /home/ce5b4deb-dc9a-4dd7-9a99-cd66f2b354ff/BGC_donor_project/data/raw
CLEAN_DIR exists: True | /home/ce5b4deb-dc9a-4dd7-9a99-cd66f2b354ff/BGC_donor_project/data/cleaned
CLEAN_DIR now exists: True


In [3]:
xlsx_files = sorted(RAW_DIR.glob("*.xlsx"))
csv_files = sorted(RAW_DIR.glob("*.csv"))

print("xlsx:", len(xlsx_files))
print("csv:", len(csv_files))

xlsx: 8
csv: 1


## Step 2 - Source Registry (Control Table)

**Purpose:** Define how each file should treated (donation vs registration) without guessing from filenames.

**What I'm looking for:** 8 entries total: each with: file_name, source_name, source_category.

**How I'll use it next:** The loop will read this registry and process each source consistently.

In [4]:
# raw filenames
[x.name for x in xlsx_files]

['Contacts.xlsx',
 'Golf 2024 teams list.xlsx',
 'Golf 2025 teams list.xlsx',
 'Great Futures 2024.xlsx',
 'Great Futures 2025 - Donation Log.xlsx',
 'Registration 2025.xlsx',
 'Signs.xlsx',
 'Sold Items.xlsx']

In [5]:
source_registry = [
    {"filename": "Golf 2024 teams list.xlsx", "source_name": "registration_golf_2024_teams_list", "source_category": "registration"},
    {"filename": "Golf 2025 teams list.xlsx", "source_name": "registration_golf_2025_teams_list", "source_category": "registration"},
    {"filename": "Great Futures 2024.xlsx", "source_name": "donation_great_futures_2024", "source_category": "donation"},
    {"filename": "Great Futures 2025 - Donation Log.xlsx", "source_name": "donation_great_futures_2025", "source_category": "donation"},
    {"filename": "Registration 2025.xlsx", "source_name": "registration_2025", "source_category": "registration"},
    {"filename": "Signs.xlsx", "source_name": "registration_signs", "source_category": "registration"},
    {"filename": "Contacts.xlsx", "source_name": "registration_contacts", "source_category": "registration"},
    {"filename": "Sold Items.xlsx", "source_name": "fundraising_auction_sold_items", "source_category": "donation"}
]

In [6]:
for s in source_registry:
    s["file_name"] = s.pop("filename")

In [7]:
source_registry[0].keys()

dict_keys(['source_name', 'source_category', 'file_name'])

In [8]:
#validation
raw_names = {p.name for p in xlsx_files}
registry_names = {s["file_name"] for s in source_registry}

missing = raw_names - registry_names
extra = registry_names - raw_names

missing, extra

(set(), set())

## Step 3 — Process One Source (Test Run)

**Purpose:** Test the cleaning pipeline on one donation source before running all 8.  

**What I’m looking for:** File loads successfully, columns standardized, metadata added, export saved.  

**How I’ll use it next:** Once the test passes, I’ll process all sources in a loop.

## Step 3A — Load one file (test)

In [9]:
test = source_registry[2]
test_path = RAW_DIR / test["file_name"]

print("Testing source:", test["source_name"], "|", test["source_category"])
print("Path exists:", test_path.exists())
print("File:", test_path.name)

df_raw = pd.read_excel(test_path)
print("Loaded shape:", df_raw.shape)
df_raw.head(3)

Testing source: donation_great_futures_2024 | donation
Path exists: True
File: Great Futures 2024.xlsx
Loaded shape: (79, 6)


,Akel,Louise,250,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,Anderson,Lawrence & Karen,250,NaN,NaN,NaN
1,Babcock,Thomas,400,NaN,NaN,NaN
2,Baxter,Robert,200,NaN,NaN,NaN


**Test load result:** The file loaded successfully and the preview looks reasonable.  

**Next step:** Create a working copy and apply “safe standardization” (column names + blanks → null + trim).

## Step 3B — Safe standardization (test file)

**Purpose:** Apply cleaning steps that are safe across all sources.  

**What I’m doing:** Standardize column names, convert blank-like values to null, and trim whitespace.  

**Why it matters:** This makes later merging and matching consistent and reduces hidden “fake missing values.”


In [10]:
df = df_raw.copy(deep=True)

# 1) Standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"\s+", "_", regex=True)
      .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# 2) Convert blank-like strings to missing
blank_like = ["", " ", "  ", "n/a", "na", "none", "null", "unknown", "-", "--"]
df = df.replace(blank_like, pd.NA)

# 3) Trim white space for text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].apply(lambda s: s.astype("string").str.strip())

# 4) metadata
df["source_system"] = test["source_name"]
df["source_category"] = test["source_category"]

print("After standardization shape:", df.shape)
df.head(3)

After standardization shape: (79, 8)


,akel,louise,NaN,unnamed_3,unnamed_4,unnamed_5,source_system,source_category
0,Anderson,Lawrence & Karen,250,<NA>,<NA>,<NA>,donation_great_futures_2024,donation
1,Babcock,Thomas,400,<NA>,<NA>,<NA>,donation_great_futures_2024,donation
2,Baxter,Robert,200,<NA>,<NA>,<NA>,donation_great_futures_2024,donation


**Result:** Column names are standardized, blank-like values are now treated as true missing values, and text fields are trimmed.  

**Quick check:** Dataset still has 79 rows (no unintended row drops).  

**Next step:** Measure missingness and confirm donation fields (date/amount) can be parsed.

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   akel             61 non-null     string
 1   louise           61 non-null     string
 2   nan              62 non-null     string
 3   unnamed_3        1 non-null      string
 4   unnamed_4        2 non-null      string
 5   unnamed_5        1 non-null      string
 6   source_system    79 non-null     object
 7   source_category  79 non-null     object
dtypes: object(2), string(6)
memory usage: 5.1+ KB


In [12]:
df.head(3)

,akel,louise,NaN,unnamed_3,unnamed_4,unnamed_5,source_system,source_category
0,Anderson,Lawrence & Karen,250,<NA>,<NA>,<NA>,donation_great_futures_2024,donation
1,Babcock,Thomas,400,<NA>,<NA>,<NA>,donation_great_futures_2024,donation
2,Baxter,Robert,200,<NA>,<NA>,<NA>,donation_great_futures_2024,donation


In [13]:
df.columns

Index([           'akel',          'louise',               nan,
             'unnamed_3',       'unnamed_4',       'unnamed_5',
         'source_system', 'source_category'],
      dtype='object')

In [14]:
# Inspecting unnamed columns
unnamed_cols = [c for c in df.columns if str(c).startswith("unnamed") or pd.isna(c)]
unnamed_cols

[nan, 'unnamed_3', 'unnamed_4', 'unnamed_5']

In [15]:
df[unnamed_cols].head(10)

,NaN,unnamed_3,unnamed_4,unnamed_5
0,250,<NA>,<NA>,<NA>
1,400,<NA>,<NA>,<NA>
2,200,<NA>,<NA>,<NA>
3,100,<NA>,<NA>,<NA>
4,500,<NA>,<NA>,<NA>
5,250,<NA>,<NA>,<NA>
6,100,<NA>,<NA>,<NA>
7,10,<NA>,<NA>,<NA>
8,2000,<NA>,<NA>,<NA>
9,150,<NA>,<NA>,<NA>


In [16]:
# drop empty columns
df[unnamed_cols].isna().mean().sort_values(ascending=False)

unnamed_3    0.987342
unnamed_5    0.987342
unnamed_4    0.974684
NaN          0.215190
dtype: float64

In [17]:
# Numbers confirmation
df[df.columns[df.columns.isna()]].describe()

,NaN
count,62
unique,18
top,100
freq,14


In [18]:
df[df.columns[df.columns.isna()]].head(10)

,NaN
0,250
1,400
2,200
3,100
4,500
5,250
6,100
7,10
8,2000
9,150


In [19]:
# Rename NaN column
nan_cols = [c for c in df.columns if pd.isna(c)]
nan_cols

[nan]

In [20]:
df = df.rename(columns={nan_cols[0]: "donation_amount"})

In [21]:
df.columns

Index(['akel', 'louise', 'donation_amount', 'unnamed_3', 'unnamed_4',
       'unnamed_5', 'source_system', 'source_category'],
      dtype='object')

In [22]:
# drop empty unnamed columns
unnamed_cols = [c for c in df.columns if str(c).startswith("unnamed")]

# keep only the ones that are 100% missing
to_drop = [c for c in unnamed_cols if df[c].isna().mean() == 1.0]

df = df.drop(columns=to_drop)

print("Dropped:", to_drop)
print("New shape:", df.shape)

Dropped: []
New shape: (79, 8)


**Result:** Removed empty “Unnamed” columns that came from blank Excel headers.  

**Why this matters:** Extra blank columns can break merges and create confusion during EDA.  

**Next step:** Export this cleaned test source to `data/cleaned/` using the naming convention.


## Step 3C — Export Cleaned Test Source

**Purpose:** Save the cleaned version of this source as a standardized CSV for later combining.  

**Naming rule:** cleaned_<category>_source##_<source_name>.csv  

**Output location:** data/cleaned/


In [23]:
source_num = 4
export_name = f"cleaned_{test['source_category']}_source{source_num:02d}_{test['source_name']}.csv"

export_path = CLEAN_DIR / export_name
df.to_csv(export_path, index=False)

export_path, export_path.exists()

(PosixPath('/home/ce5b4deb-dc9a-4dd7-9a99-cd66f2b354ff/BGC_donor_project/data/cleaned/cleaned_donation_source04_donation_great_futures_2024.csv'),
 True)

## Step 4 — Create a Reusable Processing Function

**Purpose:** Convert the working test steps into a function so every file is cleaned the same way.  

**Function output:** A small summary (rows in/out + export file path).  

**Safety rule:** Test on one file first, then loop all sources.

## Step 4A — Define process_one_file()

Logic:
1) Load raw file
2) Standardize column names
3) Convert blank-like values
4) Rename NaN column ONLY if donation source
5) Drop empty unnamed columns
6) Add metadata
7) Export using naming standard

In [24]:
def process_one_file(source_dict, source_index):
    
    file_name = source_dict["file_name"]
    source_name = source_dict["source_name"]
    source_category = source_dict["source_category"]
    
    file_path = RAW_DIR / file_name
    
    # Load
    df_raw = pd.read_excel(file_path)
    df = df_raw.copy(deep=True)
    
    # Standardize column names
    df.columns = (
        df.columns
          .astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    
    # Convert blank-like values
    blank_like = ["", " ", "  ", "n/a", "na", "none", "null", "unknown", "-", "--"]
    df = df.replace(blank_like, pd.NA)
    
    # Trim whitespace
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    df[text_cols] = df[text_cols].apply(lambda s: s.astype("string").str.strip())
    
    # Rename NaN header if donation source
    if source_category == "donation":
        nan_cols = [c for c in df.columns if c == "nan"]
        if len(nan_cols) == 1:
            df = df.rename(columns={nan_cols[0]: "donation_amount"})
    
    # Drop empty unnamed columns
    unnamed_cols = [c for c in df.columns if c.startswith("unnamed")]
    to_drop = [c for c in unnamed_cols if df[c].isna().mean() == 1.0]
    df = df.drop(columns=to_drop)
    
    # Add metadata
    df["source_system"] = source_name
    df["source_category"] = source_category
    
    # Export
    export_name = f"cleaned_{source_category}_source{source_index:02d}_{source_name}.csv"
    export_path = CLEAN_DIR / export_name
    df.to_csv(export_path, index=False)
    
    return {
        "source_index": source_index,
        "source_name": source_name,
        "rows_in": len(df_raw),
        "rows_out": len(df),
        "export_file": export_name
    }

## Step 4B — Test process_one_file() on One Donation Source

Purpose:
Confirm the function reproduces the same result as my manual test.

Success criteria:
- No errors
- Export file created in data/cleaned/
- rows_in matches expected
- rows_out matches expected

In [26]:
test_summary = process_one_file(source_registry[3], 4)
test_summary

{'source_index': 4,
 'source_name': 'donation_great_futures_2025',
 'rows_in': 69,
 'rows_out': 69,
 'export_file': 'cleaned_donation_source04_donation_great_futures_2025.csv'}

In [27]:
(CLEAN_DIR / test_summary["export_file"]).exists()

True

## Step 5 — Process All Sources (Batch Run)

Purpose:
Run the standardized cleaning pipeline across all 8 Excel sources.

Output:
- 8 cleaned CSVs saved to data/cleaned/
- A processing summary table (rows in/out + export names)

Safety:
If any file fails, stop and fix that source, then rerun.

In [28]:
results = []

for i, src in enumerate(source_registry, start=1):
    summary = process_one_file(src, i)
    results.append(summary)

results_df = pd.DataFrame(results)
results_df


,source_index,source_name,rows_in,rows_out,export_file
0,1,registration_golf_2024_teams_list,147,147,cleaned_registration_source01_registration_gol...
1,2,registration_golf_2025_teams_list,137,137,cleaned_registration_source02_registration_gol...
2,3,donation_great_futures_2024,79,79,cleaned_donation_source03_donation_great_futur...
3,4,donation_great_futures_2025,69,69,cleaned_donation_source04_donation_great_futur...
4,5,registration_2025,125,125,cleaned_registration_source05_registration_202...
5,6,registration_signs,38,38,cleaned_registration_source06_registration_sig...
6,7,registration_contacts,830,830,cleaned_registration_source07_registration_con...
7,8,fundraising_auction_sold_items,128,128,cleaned_donation_source08_fundraising_auction_...


## Step 6 — Data Quality Notes (Donation Sources)

**Purpose:**
Summarize missingness and potential duplicates at a high level using cleaned exports.

**Goal:**
Create quick notes for stakeholders and guide Week 2 donor matching.

In [29]:
donation_files = sorted(CLEAN_DIR.glob("cleaned_donation_*.csv"))
len(donation_files), [p.name for p in donation_files]

(4,
 ['cleaned_donation_source03_donation_great_futures_2024.csv',
  'cleaned_donation_source04_donation_great_futures_2024.csv',
  'cleaned_donation_source04_donation_great_futures_2025.csv',
  'cleaned_donation_source08_fundraising_auction_sold_items.csv'])

In [30]:
donation_dfs = []
for p in donation_files:
    tmp = pd.read_csv(p)
    tmp["cleaned_file"] = p.name
    donation_dfs.append(tmp)

donations_all = pd.concat(donation_dfs, ignore_index=True)
donations_all.shape

(355, 21)

In [31]:
donations_all.duplicated().sum()

38

**Data quality snapshot (donation sources):**

- Missingness varies by field (top missing fields listed above).

- Full-row duplicates count recorded above.

- Next: create donor-level identity rules + review list (Week 2).


## Questions for Mentor / Stakeholder (Week 1)

1) In event spreadsheets, does "Paid = X" represent:


   - registration fee / ticket purchase
   - donation
   - both (ticket + optional donation)


3) What should be considered the "source of truth" for donor contact fields?
 


    - email vs phone vs address priority?



5) Should registration-only participants be included in the donor database?



    - If yes: should they be labeled as "non-donor contact" until they donate?



7) Are there any rules for how households should be handled?




     - same address different names → same donor household or separate donors?



9) What donation fields are required to be valid for analysis?
 


   - amount required? date required? payment method required?



11) What is the preferred reporting grain for leadership?


   - donor-level (1 row per donor)
   - donation-level (1 row per donation)
   - both (recommended)

### Data Quality Note (Week 1)

- Full-row duplicates exist in the combined donation extracts.

- Interpretation: same donation or donor record may have been entered more than once across sources.

- Action for Week 2: build a possible-duplicates review list + dedupe rules ("do no harm" + stakeholder verification).
